# 05 · Classification benchmark and diagnostics

Can full-session attributes distinguish recorded purchases? We compare a prior-only baseline, regularized logistic regression, and a random forest. The selected model is determined on training CV only and evaluated on group-isolated held-out sessions.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from scripts.model_purchase_intent import load_model_data, split_sessions, FEATURES

ROOT = Path.cwd()
SOURCE = ROOT / 'data/raw/online_shoppers_intention.csv'
if not SOURCE.exists():
    raise FileNotFoundError(SOURCE)
X, y, groups = load_model_data(pd.read_csv(SOURCE))
train, test = split_sessions(y, groups)
assert not set(groups[train]).intersection(groups[test])
assert 'PageValues' not in X.columns
print(f"Train {len(train):,} ({int(y[train].sum()):,} purchases) | Test {len(test):,} ({int(y[test].sum()):,} purchases)")
print('Principal predictors:', ', '.join(FEATURES))

## Modeling design

The split isolates identical measured input profiles. A three-fold stratified group-aware cross-validation on **training sessions only** selects hyperparameters by average precision. The test set is not used for tuning. One-hot encoded categories and numeric scaling are fitted within each training fold.

In [ ]:
model_dir = ROOT / 'outputs/modeling'
if not (model_dir / 'metrics.json').is_file():
    raise FileNotFoundError('Run python -m scripts.run_upgrade before this notebook')
result = json.loads((model_dir/'metrics.json').read_text(encoding='utf-8'))
assert result['test_sessions'] == len(test)
assert result['training_sessions'] == len(train)
print('CV-selected model:', result['chosen_model'])
print('Training cross-validated average precision:', round(result['selected_cv_average_precision'],3))
display(pd.read_csv(model_dir/'tables/cross_validation_tuning.csv'))

## Held-out results

Average precision is compared against the prior-only no-skill prevalence baseline; classification performance is not a guarantee of actual business benefit.

In [ ]:
display(pd.read_csv(model_dir/'tables/model_comparison.csv').round(3))
display(pd.read_csv(model_dir/'tables/threshold_sensitivity.csv')[['policy','threshold','precision','recall','f1']].round(3))
print('Training-selected threshold:', round(result['train_cv_selected_threshold'], 3))

## Transaction-linked-feature sensitivity

`PageValues` is intentionally excluded from the principal model. Including a variable defined using transaction-related values is a possible leakage route for prospective prediction.

In [ ]:
display(pd.read_csv(model_dir/'tables/pagevalues_sensitivity.csv')[['feature_set','average_precision','roc_auc']].round(3))

## Evaluation visualizations

In [ ]:
from IPython.display import Image, display as show
for image_name in ('precision_recall.png','confusion_matrix.png','confusion_matrix_cv_threshold.png','permutation_importance.png'):
    image_path = model_dir / 'charts' / image_name
    if not image_path.exists():
        raise FileNotFoundError(image_path)
    show(Image(filename=str(image_path), width=750))

**Conclusion:** Retrospective discrimination of complete sessions is not validated real-time intent prediction. We have neither time-indexed event features nor external or forward-time validation. Permutation importance is predictive association, not causal attribution.